In [15]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import datetime

In [16]:
# Importing the files

# Data consisting of the purchases of each customer for each date
transactions = pd.read_csv("data/transactions.csv")
# Detailed metadata for each article_id available for purchase
articles = pd.read_csv("data/articles.csv")
# Metadata for each customer_id in dataset
customers = pd.read_csv("data/customers.csv")

In [17]:
articles.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105542 entries, 0 to 105541
Data columns (total 25 columns):
 #   Column                        Non-Null Count   Dtype 
---  ------                        --------------   ----- 
 0   article_id                    105542 non-null  int64 
 1   product_code                  105542 non-null  int64 
 2   prod_name                     105542 non-null  object
 3   product_type_no               105542 non-null  int64 
 4   product_type_name             105542 non-null  object
 5   product_group_name            105542 non-null  object
 6   graphical_appearance_no       105542 non-null  int64 
 7   graphical_appearance_name     105542 non-null  object
 8   colour_group_code             105542 non-null  int64 
 9   colour_group_name             105542 non-null  object
 10  perceived_colour_value_id     105542 non-null  int64 
 11  perceived_colour_value_name   105542 non-null  object
 12  perceived_colour_master_id    105542 non-null  int64 
 13 

In [18]:
customers.info()

transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1371980 entries, 0 to 1371979
Data columns (total 7 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   customer_id             1371980 non-null  object 
 1   FN                      476930 non-null   float64
 2   Active                  464404 non-null   float64
 3   club_member_status      1365918 non-null  object 
 4   fashion_news_frequency  1355969 non-null  object 
 5   age                     1356119 non-null  float64
 6   postal_code             1371980 non-null  object 
dtypes: float64(3), object(4)
memory usage: 73.3+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1224824 entries, 0 to 1224823
Data columns (total 5 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   t_dat             1224824 non-null  object 
 1   customer_id       1224824 non-null  object 
 2   article_id        1224824 n

In [ ]:
customers = pd.read_csv("data/customers.csv")
articles = pd.read_csv("data/articles.csv")
transactions = pd.read_csv("data/transactions.csv")

# Handle nulls in customers
customers['FN'] = customers['FN'].fillna(0)
customers['Active'] = customers['Active'].fillna(0)
customers['age'] = customers['age'].fillna(customers['age'].median())
customers['club_member_status'] = customers['club_member_status'].fillna('UNKNOWN')
customers['fashion_news_frequency'] = customers['fashion_news_frequency'].fillna('NONE')

# Convert date in transactions
transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])

# Create user features
customers['age_group'] = pd.cut(customers['age'], bins=[0, 20, 30, 40, 50, 60, 100], labels=['&lt;20', '20-30', '30-40', '40-50', '50-60', '60+'])

# One-hot encode categorical user features
user_features = pd.get_dummies(customers[['club_member_status', 'fashion_news_frequency', 'age_group', 'FN', 'Active']], drop_first=True)

# Merge back, keep customer_id
customers = pd.concat([customers[['customer_id']], user_features], axis=1)

# Item features: select key categorical columns
item_categorical_cols = ['product_type_name', 'colour_group_name', 'department_name', 'index_group_name', 'section_name', 'garment_group_name']

# One-hot encode (but since many categories, perhaps use label encoding or select top)
# For simplicity, use get_dummies, but it will create many columns
item_features = pd.get_dummies(articles[item_categorical_cols], drop_first=True)

# Add article_id
item_features = pd.concat([articles[['article_id']], item_features], axis=1)

# Interaction features
# User purchase count
user_purchase_count = transactions.groupby('customer_id').size().reset_index(name='purchase_count')

# Item popularity
item_popularity = transactions.groupby('article_id').size().reset_index(name='popularity')

# Average price per user
user_avg_price = transactions.groupby('customer_id')['price'].mean().reset_index(name='avg_price')

# Last purchase date per user
user_last_purchase = transactions.groupby('customer_id')['t_dat'].max().reset_index(name='last_purchase')

# Recency (days since last purchase, assuming current date is max date)
max_date = transactions['t_dat'].max()
user_last_purchase['recency'] = (max_date - user_last_purchase['last_purchase']).dt.days

# Merge user features
customers = customers.merge(user_purchase_count, on='customer_id', how='left').fillna(0)
customers = customers.merge(user_avg_price, on='customer_id', how='left').fillna(0)
customers = customers.merge(user_last_purchase[['customer_id', 'recency']], on='customer_id', how='left').fillna(999)

# For items, merge popularity
articles = articles.merge(item_popularity, on='article_id', how='left').fillna(0)